In [ ]:
# Resstock model download, and ami optimization 

"""
Created on Sep 11 16:00:00 2025

@author: Tanushree Charan

"""

In [ ]:
pip install awscli

In [ ]:
pip install pyarrow fastparquet

In [ ]:
!aws --version

In [ ]:
pip install pandas

In [ ]:
import subprocess
import zipfile
import os
import shutil
import json
import pandas as pd
from pathlib import Path

In [ ]:
# SCRIPT TO READ AND SAVE ALL BUILDING IDS FROM THE NY METADATA FILE (FILTERED FOR DRYDEN RELEVANT COUNTIES)
# Read list of all building IDs

# path to metadata parquet

current_path = Path(os.getcwd())
data_path = current_path / '..'/ 'data' 
buildstock_metadata_path = data_path / 'NY_baseline_metadata_and_annual_results.xlsx'

# Read the "select_counties" sheet
df = pd.read_excel(buildstock_metadata_path, sheet_name="select_counties")

# Number of rows in the DataFrame
num_rows = len(df)
print("Number of rows:", num_rows)

building_ids_metadata = df["bldg_id"]

# Save building IDs to CSV
output_path = data_path / "base_files" / "building_ids_metadata.csv"
building_ids_metadata.to_csv(output_path, index=False)

print("bldg_id count:", len(building_ids_metadata))
print(building_ids_metadata)
print('done')

In [ ]:
# SCRIPT TO READ AND STORE ALL BUILDING IDS FOR THE NY METADATA FILE (FILTERED FOR DRYDEN RELEVANT COUNTIES)
# SCRIPT STORES LIST OF BUILDING IDS
# read metadata IDs
current_path = Path(os.getcwd())
data_path = current_path / '..'/ 'data'
path = data_path / "base_files" / "building_ids_metadata.csv"
df = pd.read_csv(path)
building_ids = df["bldg_id"]
print("bldg_id count:", len(building_ids))

In [ ]:
# MAC SCRIPT TO DOWNLOAD RESSTOCK MODELS FROM AWS BASED ON INPUT MODEL IDS

# List of building IDs to download

#building_ids = [140567, 253852, 379341, 335998, 43442, 491645, 533465, 275002, 295328, 302371, 303300, 303512, 310716, 269317, 273946, 329817, 350503, 364293, 65865, 421718, 438492, 444781, 115980, 88748, 458439]

building_ids = [140567, 253852, 379341]
    
# Base S3 path for ResStock 2024 release
s3_base = (
    "s3://oedi-data-lake/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2024/"
    "resstock_tmy3_release_2/model_and_schedule_files/"
    "building_energy_models/upgrade=0/"
)

# base files
base_files = Path("../data/base_files")
measures_og = Path(base_files / "measures")
workflow_og = Path(base_files / "workflow_resstock.osw")
# weather file
weather_file = "Ithaca_2024.epw"
weather_og = Path(base_files / "weather" / weather_file)

# Output directories
extract_dir = Path("../data/resstock_models")
extract_dir.mkdir(exist_ok=True)

# Loop through each ID and download + unzip
for bid in building_ids:
    
    padded_id = f"{bid:07d}"
    filename = f"bldg{padded_id}-up00.zip"
    s3_path = s3_base + filename

    local_extract_path = extract_dir / f"bldg{padded_id}"
    zip_file = local_extract_path / filename
    # make models folder
    local_extract_path_models = extract_dir / f"bldg{padded_id}" / "models"

    ### DOWNLOAD MODELS
    print(f"Downloading {filename}...")
    try:
        subprocess.run(
            ["aws", "s3", "cp", s3_path, str(zip_file), "--no-sign-request"],
            check=True
        )
    except subprocess.CalledProcessError as e:
        print(f"Failed to download {filename}: {e}")
        continue

    print(f"Unzipping {filename} to folder {local_extract_path}...")
    try:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            local_extract_path.mkdir(exist_ok=True)
            local_extract_path_models.mkdir(exist_ok=True)

            zip_ref.extractall(local_extract_path_models)
    except zipfile.BadZipFile as e:
        print(f"Bad ZIP file: {filename}: {e}")
        continue
        
    ### WEATHER
    # create weather folder if it doesn't exist
    weather_new = os.path.join(local_extract_path, 'weather')
    os.makedirs(weather_new, exist_ok=True)  # Create folder using filename if it doesn't exist
    shutil.copy(weather_og, weather_new)

    ### MEASURES
    # copy default measures
    new_measure_path = os.path.join(local_extract_path, 'measures')

    # Check if the folder already exists
    if os.path.exists(new_measure_path):
        # If the folder already exists, delete it
        try:
            shutil.rmtree(new_measure_path)  # Delete the existing folder
            print(f"Deleted existing folder at: {new_measure_path}")
        except Exception as e:
            print(f"Error deleting folder: {e}")
            raise

    # Copy the contents of the measure_path to new_measure_path
    try:
        shutil.copytree(measures_og, new_measure_path)
        print("All folders and files copied successfully.")
    except Exception as e:
        print(f"Error copying files: {e}")


    #### OSW FILE
    new_osw_path = os.path.join(local_extract_path, 'workflow_resstock.osw')
    shutil.copy(workflow_og, new_osw_path)

    with open(new_osw_path, 'r') as osw:
        data = json.load(osw)
        data['seed_file'] =  "in.osm"
        data['weather_file'] = f"{weather_file}"

    with open(new_osw_path, 'w') as osw:
        json.dump(data, osw, indent=2)


    # Optional: remove the ZIP file after extraction
    #zip_file.unlink()

print("Done.")


In [ ]:
# WINDOWS VIRTUAL MACHINE SCRIPT TO DOWNLOAD RESSTOCK MODELS FROM AWS BASED ON INPUT MODEL IDS
# --- ResStock downloader & prep (Windows-safe) ---

import os
import json
import zipfile
import sys
import subprocess
import shutil
from pathlib import Path
from typing import Iterable

# -------------------
# CONFIGURE HERE
# -------------------

# Building IDs to fetch


# Base S3 path for ResStock 2024 release
S3_BASE = (
    "s3://oedi-data-lake/nrel-pds-building-stock/"
    "end-use-load-profiles-for-us-building-stock/2024/"
    "resstock_tmy3_release_2/model_and_schedule_files/"
    "building_energy_models/upgrade=0/"
)

# Local base files (weather, measures, workflow)
BASE_FILES = Path("../data/base_files")
MEASURES_OG = BASE_FILES / "measures"
WORKFLOW_OG = BASE_FILES / "workflow_resstock.osw"
WEATHER_FILE = "Ithaca_2024.epw"          # <-- change if needed
WEATHER_OG = BASE_FILES / "weather" / WEATHER_FILE

# Output directory where each bldg folder will live
EXTRACT_DIR = Path("../data/resstock_models")

# -------------------
# AWS CLI LAUNCHER
# -------------------

# Always invoke AWS CLI via the running Python (avoids PATH/.py-association issues on Windows)
AWS_PREFIX = [sys.executable, "-m", "awscli"]

def aws_cp(s3_path: str, local_path: Path) -> None:
    """Copy from S3 to local path using awscli module; ensures parent dir exists."""
    local_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = AWS_PREFIX + ["s3", "cp", s3_path, str(local_path), "--no-sign-request"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

# -------------------
# UTILITIES
# -------------------

def ensure_base_files() -> None:
    missing = []
    if not MEASURES_OG.exists():
        missing.append(str(MEASURES_OG))
    if not WORKFLOW_OG.exists():
        missing.append(str(WORKFLOW_OG))
    if not WEATHER_OG.exists():
        missing.append(str(WEATHER_OG))
    if missing:
        raise FileNotFoundError(
            "Missing required base files:\n- " + "\n- ".join(missing)
        )

def unzip_to_dir(zip_path: Path, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

def copy_weather_and_measures(local_extract_path: Path) -> None:
    # Weather
    weather_dst = local_extract_path / "weather"
    weather_dst.mkdir(exist_ok=True)
    shutil.copy(WEATHER_OG, weather_dst)

    # Measures (replace if exists)
    measures_dst = local_extract_path / "measures"
    if measures_dst.exists():
        shutil.rmtree(measures_dst)
        print(f"Deleted existing folder: {measures_dst}")
    shutil.copytree(MEASURES_OG, measures_dst)

def patch_osw(local_extract_path: Path) -> None:
    osw_dst = local_extract_path / "workflow_resstock.osw"
    shutil.copy(WORKFLOW_OG, osw_dst)
    with open(osw_dst, "r") as f:
        data = json.load(f)
    # Minimal edits (adjust if your workflow expects different keys)
    data["seed_file"] = "in.osm"
    data["weather_file"] = WEATHER_FILE
    with open(osw_dst, "w") as f:
        json.dump(data, f, indent=2)

# -------------------
# MAIN
# -------------------

def main():
    print("Verifying base files...")
    ensure_base_files()
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    # Optional sanity check: show aws version via module
    try:
        subprocess.run(AWS_PREFIX + ["--version"], check=True)
    except Exception as e:
        print("Warning: unable to run 'python -m awscli --version':", e)

    completed, failed = [], []

    for bid in building_ids:
        padded = f"{bid:07d}"
        filename = f"bldg{padded}-up00.zip"
        s3_path = S3_BASE + filename

        local_extract_path = EXTRACT_DIR / f"bldg{padded}"
        models_dir = local_extract_path / "models"
        zip_file = local_extract_path / filename

        print(f"\n=== {bid} :: downloading {filename} ===")
        try:
            aws_cp(s3_path, zip_file)
        except subprocess.CalledProcessError as e:
            print(f"Failed to download {filename}: {e}")
            failed.append((bid, "download"))
            continue

        print(f"Unzipping to {models_dir} ...")
        try:
            unzip_to_dir(zip_file, models_dir)
        except zipfile.BadZipFile as e:
            print(f"Bad ZIP for {filename}: {e}")
            failed.append((bid, "bad_zip"))
            continue

        try:
            copy_weather_and_measures(local_extract_path)
            patch_osw(local_extract_path)
        except Exception as e:
            print(f"Post-processing error for {bid}: {e}")
            failed.append((bid, "post_process"))
            continue

        # Optional: remove the ZIP to save space
        # try:
        #     zip_file.unlink(missing_ok=True)
        # except Exception as e:
        #     print(f"Could not delete {zip_file}: {e}")

        completed.append(bid)
        print(f"Done {bid}")

    print("\nSUMMARY")
    print("-------")
    print(f"Completed: {len(completed)}  {completed}")
    if failed:
        print(f"Failed: {len(failed)}  {failed}")
    else:
        print("Failed: 0")

if __name__ == "__main__":
    main()


In [ ]:
# --- Faster ResStock downloader: parallel + multipart S3 ---
import os, json, zipfile, sys, subprocess, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========================
# CONFIG
# ========================
BUILDING_IDS = [140567, 253852, 379341]   # <--- edit
S3_BASE = ("s3://oedi-data-lake/nrel-pds-building-stock/"
           "end-use-load-profiles-for-us-building-stock/2024/"
           "resstock_tmy3_release_2/model_and_schedule_files/"
           "building_energy_models/upgrade=0/")

BASE_FILES = Path("../data/base_files")
MEASURES_OG = BASE_FILES / "measures"
WORKFLOW_OG = BASE_FILES / "workflow_resstock.osw"
WEATHER_FILE = "Ithaca_2024.epw"
WEATHER_OG = BASE_FILES / "weather" / WEATHER_FILE

EXTRACT_DIR = Path("../data/resstock_models")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 6          # try 4–8 depending on your bandwidth/CPU
CHUNK_MB = 8             # multipart chunk size for S3 (MB)

# ========================
# S3 DOWNLOAD IMPL (boto3 if available -> fast; else awscli module)
# ========================
_use_boto3 = False
try:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    from boto3.s3.transfer import TransferConfig, S3Transfer
    _use_boto3 = True
    _s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
    _transfer = S3Transfer(
        _s3,
        TransferConfig(
            multipart_threshold=CHUNK_MB*1024*1024,
            multipart_chunksize=CHUNK_MB*1024*1024,
            max_concurrency=10,
            use_threads=True
        )
    )
except Exception:
    pass

AWS_PREFIX = [sys.executable, "-m", "awscli"]  # fallback

def s3_url_parts(s3_url: str):
    # "s3://bucket/key..." -> ("bucket","key")
    assert s3_url.startswith("s3://")
    rest = s3_url[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def download_s3(s3_url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if _use_boto3:
        bucket, key = s3_url_parts(s3_url)
        _transfer.download_file(bucket, key, str(dest))
    else:
        cmd = AWS_PREFIX + ["s3", "cp", s3_url, str(dest), "--no-sign-request", "--only-show-errors"]
        subprocess.run(cmd, check=True)

# ========================
# UTILITIES
# ========================
def ensure_base_files():
    for p in [MEASURES_OG, WORKFLOW_OG, WEATHER_OG]:
        if not p.exists():
            raise FileNotFoundError(f"Missing base file/dir: {p}")

def unzip_to_dir(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

def copy_weather_and_measures(local_extract_path: Path):
    (local_extract_path / "weather").mkdir(exist_ok=True)
    shutil.copy(WEATHER_OG, local_extract_path / "weather")

    measures_dst = local_extract_path / "measures"
    if measures_dst.exists():
        shutil.rmtree(measures_dst)
    shutil.copytree(MEASURES_OG, measures_dst)

def patch_osw(local_extract_path: Path):
    osw_dst = local_extract_path / "workflow_resstock.osw"
    shutil.copy(WORKFLOW_OG, osw_dst)
    with open(osw_dst, "r") as f:
        data = json.load(f)
    data["seed_file"] = "in.osm"
    data["weather_file"] = WEATHER_FILE
    with open(osw_dst, "w") as f:
        json.dump(data, f, indent=2)

def building_done(local_extract_path: Path) -> bool:
    # crude skip: consider done if models dir exists and has files
    models = local_extract_path / "models"
    return models.exists() and any(models.iterdir())

def process_building(bid: int):
    padded = f"{bid:07d}"
    filename = f"bldg{padded}-up00.zip"
    s3_url = S3_BASE + filename
    local_extract_path = EXTRACT_DIR / f"bldg{padded}"
    models_dir = local_extract_path / "models"
    zip_file = local_extract_path / filename

    if building_done(local_extract_path):
        return bid, "skipped"

    try:
        download_s3(s3_url, zip_file)
        unzip_to_dir(zip_file, models_dir)
        copy_weather_and_measures(local_extract_path)
        patch_osw(local_extract_path)
        # zip_file.unlink(missing_ok=True)  # optionally delete zip
        return bid, "ok"
    except zipfile.BadZipFile as e:
        return bid, f"bad_zip: {e}"
    except subprocess.CalledProcessError as e:
        return bid, f"download_fail: {e}"
    except Exception as e:
        return bid, f"error: {e}"

# ========================
# MAIN (parallel)
# ========================
def main():
    ensure_base_files()
    print(f"Using downloader: {'boto3 (multipart)' if _use_boto3 else 'awscli module'}")
    results = {"ok": [], "skipped": [], "failed": []}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_building, bid): bid for bid in BUILDING_IDS}
        for fut in as_completed(futures):
            bid, status = fut.result()
            if status == "ok":
                print(f"✔ {bid} done")
                results["ok"].append(bid)
            elif status == "skipped":
                print(f"↩ {bid} skipped (already processed)")
                results["skipped"].append(bid)
            else:
                print(f"✖ {bid} {status}")
                results["failed"].append((bid, status))

    print("\nSUMMARY")
    print("-------")
    print(f"Completed: {results['ok']}")
    print(f"Skipped:   {results['skipped']}")
    print(f"Failed:    {results['failed']}")

if __name__ == "__main__":
    main()


In [ ]:
# RUN OPENSTUDIO MODELS 1 at a time 

# Windows
#openstudio_path="C:/openstudio-3.9.0/bin/openstudio.exe"

# Mac
openstudio_path="/Applications/OpenStudio-3.9.0/bin/openstudio"


# Check if OpenStudio is installed and accessible
try:
    result = subprocess.run([openstudio_path, "openstudio_version"], capture_output=True, text=True, check=True)
    print(f"OpenStudio is installed at {openstudio_path}.")

except subprocess.CalledProcessError:
    print(f"OpenStudio is not installed or not accessible at {openstudio_path}.")

# Loop through each ID and download + unzip
for bid in building_ids:
    
    padded_id = f"{bid:07d}"
    filename = f"bldg{padded_id}-up00.zip"
    
    local_extract_path = extract_dir / f"bldg{padded_id}"
    
    osw = Path(os.path.join(local_extract_path, 'workflow_resstock.osw'))
    
    # Run the OpenStudio command
    # TO DO: Run 15 min timestep
    result = subprocess.run([openstudio_path, "run", "-w", osw], capture_output=True, text=True)
    print(result)
    # Assert that there are no errors
    assert result.returncode == 0, f"OpenStudio run failed for {bid}: {result.stderr}"
    assert result.stderr == "", f"Unexpected stderr output for {bid}: {result.stderr}"

### RUN MODELS IN PARALLEL

In [ ]:
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# Windows
openstudio_path="C:/openstudio-3.9.0/bin/openstudio.exe"

# Mac
#openstudio_path="/Applications/OpenStudio-3.9.0/bin/openstudio"

# Quick check
subprocess.run([openstudio_path, "openstudio_version"], check=True, capture_output=True, text=True)

MAX_PARALLEL = 50
extract_dir = Path("../data/resstock_models")

def run_one(bid: int):
    padded = f"{bid:07d}"
    osw = Path(extract_dir) / f"bldg{padded}" / "workflow_resstock.osw"
    if not osw.exists():
        return bid, 1, f"Missing OSW: {osw}"
    r = subprocess.run([openstudio_path, "run", "-w", str(osw)], capture_output=True, text=True)
    return bid, r.returncode, r.stderr.strip()

print(f"Running {len(building_ids)} models (up to {MAX_PARALLEL} at once)...")
errors = []

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    # start all jobs at once (up to MAX_PARALLEL run concurrently)
    futures = [ex.submit(run_one, bid) for bid in building_ids]

    # handle results as each job finishes
    for i, fut in enumerate(as_completed(futures), 1):
        bid, rc, err = fut.result()        # what run_one() returns
        ok = (rc == 0 and err == "")       # success if no error and rc==0
        print(f"[{i}/{len(building_ids)}] bldg {bid}: {'OK' if ok else 'FAIL'}")
        if not ok:
            errors.append((bid, rc, err))

# after all jobs finish, fail the cell if any failed
if errors:
    msg = " | ".join([f"{bid}: rc={rc}, err={err[:120]}" for bid, rc, err in errors])
    raise AssertionError(f"OpenStudio failed for {len(errors)} model(s): {msg}")

print("All done")


In [ ]:
# RESSTOCK DATA PREPARATION FOR OPTIMIZATION
# ## SCRIPT FOR GENERATING 15 MIN, 1 HOUR PARQUET FILES FROM EPLUSOUT.CSV RESSTOCK SIMULATIONS- NON PARALLELIZED




# cols = {"id": "bldg_id",
#         "time": "timestamp",
#         "ts_elec": "electricity.total.energy_consumption",
#         "annl_elec": "out.electricity.total.energy_consumption"}


# Save in data/parquet Folder
parquet_folder = Path("../data/parquet")
parquet_folder.mkdir(exist_ok=True)

frames = []
target_cols = ["Date/Time", "EMS:EMS_ElectricityFacility_kWh [](TimeStep)"]

# find eplusout.csv file
for bldg_folder in extract_dir.iterdir():
    if bldg_folder.is_dir():
        csv_path = bldg_folder.rglob("eplusout.csv")
        csv_path = next(csv_path, None)  # get first match
        if csv_path:
            bldg_id = bldg_folder.name
            try:
                df = pd.read_csv(csv_path, usecols=target_cols)
                
            except ValueError as e:
                print(f"Required columns not found in {csv_path}: {e}")
                continue
                
            # save to parquet
            df = df.rename(
                    columns={
                        "Date/Time": "time",
                        "EMS:EMS_ElectricityFacility_kWh [](TimeStep)": "ts_elec"
                    })
            
            df["bldg_id"] = int(bldg_id.replace("bldg",""))
            # CURRENTLY TIME STEP IS 15, AGGREGATE TO 1 HOUR IF NEEDED                          
            # Step 1: Clean up spaces
            df["time"] = df["time"].str.strip().str.replace("  ", " ")
            
            # Step 2: Fix special "24:00:00" to "00:00:00"
            df["time"] = df["time"].str.replace("24:00:00", "00:00:00")
            
            # Step 3: Add the year 2024 in front
            df["time"] = "2024-" + df["time"]
            
            # Step 4: Convert to datetime
            df["time"] = pd.to_datetime(df["time"], format="%Y-%m/%d %H:%M:%S")
            frames.append(df)
        else:
            print(f"No eplusout.csv found in {bldg_folder}")


if frames:
    # TIMESERIES PARQUET FILE
    all_df = pd.concat(frames, ignore_index=True)
    all_df = all_df.sort_values(["bldg_id", "time"])
    # TIMESERIES 1 HOUR PARQUET FILE
    all_df_hour = all_df.set_index("time").groupby("bldg_id")["ts_elec"].resample("h").sum().reset_index()

    
    out_path = parquet_folder / "resstock_timeseries_all_15.parquet"
    all_df.to_csv(parquet_folder / "resstock_timeseries_all_15.csv", index=False)
    all_df.to_parquet(out_path, index=False)

    all_df_hour.to_csv(parquet_folder / "resstock_timeseries_all_hour.csv", index=False)
    all_df_hour.to_parquet(parquet_folder / "resstock_timeseries_all_hour.parquet", index=False)
    
    print(f"Saved {out_path}")


    # ANNUAL PARQUET FILE
    df_anl = all_df.groupby("bldg_id", as_index=False)['ts_elec'].sum()
    df_anl = df_anl.rename(columns={"ts_elec": "annl_elec"})
    out_path_anl = parquet_folder / "resstock_timeseries_annual.parquet"
    df_anl.to_parquet(out_path_anl, index=False)
    df_anl.to_csv(parquet_folder / "resstock_timeseries_annual.csv")
    print(f"Saved {out_path_anl}")

else:
    print("No timeseries found.")


In [ ]:
# RESSTOCK DATA PREPARATION FOR OPTIMIZATION


## SCRIPT FOR GENERATING 15 MIN, 1 HOUR PARQUET FILES FROM EPLUSOUT.CSV RESSTOCK SIMULATIONS- PARALLELIZED

# --- ResStock: Build combined 15-min, hourly, and annual tables (no per-building files) ---
# Parallel read; stream-append to combined CSVs; Parquet via pyarrow if available.

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

# ========================
# CONFIG
# ========================
extract_dir = Path("../data/resstock_models")  # where bldgXXXXXX folders live
out_dir     = Path("../data/parquet")
out_dir.mkdir(parents=True, exist_ok=True)

OUT_15_CSV   = out_dir / "resstock_timeseries_all_15.csv"
OUT_15_PQ    = out_dir / "resstock_timeseries_all_15.parquet"
OUT_HR_CSV   = out_dir / "resstock_timeseries_all_hour.csv"
OUT_HR_PQ    = out_dir / "resstock_timeseries_all_hour.parquet"
OUT_ANL_CSV  = out_dir / "resstock_timeseries_annual.csv"
OUT_ANL_PQ   = out_dir / "resstock_timeseries_annual.parquet"

YEAR_PREFIX = "2024-"
N_WORKERS   = 8  # tune for your CPU/SSD

DATE_COL = "Date/Time"
# Primary column you referenced; add alternates if some runs differ:
TS_COL_CANDIDATES = [
    "EMS:EMS_ElectricityFacility_kWh [](TimeStep)",
    # "EMS:EMS_ElectricityFacility_kWh [](Hourly)",
    # "Electricity:Facility [J](TimeStep)",  # would need J->kWh conversion
]

# Speed up CSV parsing if pyarrow is available
READ_KW = {"usecols": [DATE_COL] + TS_COL_CANDIDATES}
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    READ_KW["engine"] = "pyarrow"
    parquet_ok = True
except Exception:
    parquet_ok = False

# ------------------------
# Small I/O helpers
# ------------------------
def append_csv(df: pd.DataFrame, path: Path, columns_order: list[str]):
    path.parent.mkdir(parents=True, exist_ok=True)
    # enforce column order and append with header once
    df = df.loc[:, columns_order]
    header = not path.exists()
    df.to_csv(path, index=False, mode="a", header=header)

class StreamParquetWriter:
    """Append rows to a single Parquet file using pyarrow ParquetWriter."""
    def __init__(self, path: Path, schema_fields: list[tuple[str, pa.DataType]]):
        self.path = path
        self.schema = pa.schema([pa.field(n, t) for n, t in schema_fields])
        self.writer = None

    def write(self, df: pd.DataFrame):
        if df.empty:
            return
        tbl = pa.Table.from_pandas(df, schema=self.schema, preserve_index=False)
        if self.writer is None:
            self.writer = pq.ParquetWriter(str(self.path), schema=self.schema, compression="snappy")
        self.writer.write_table(tbl)

    def close(self):
        if self.writer is not None:
            self.writer.close()

# ------------------------
# Parsing helpers
# ------------------------
def find_bldg_id_from_path(p: Path) -> int | None:
    for parent in [p] + list(p.parents):
        name = parent.name
        if name.startswith("bldg") and name[4:].isdigit():
            return int(name[4:])
    return None

def pick_ts_column(df_cols) -> str | None:
    for c in TS_COL_CANDIDATES:
        if c in df_cols:
            return c
    return None

def parse_time_col(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip().str.replace(r"\s{2,}", " ", regex=True)
    s = s.str.replace("24:00:00", "00:00:00", regex=False)
    return pd.to_datetime(YEAR_PREFIX + s, format="%Y-%m/%d %H:%M:%S", errors="coerce")

# ------------------------
# Per-file processing
# ------------------------
def process_csv(csv_path: Path):
    """Returns (bldg_id, df15, df_hr, ann_kwh, status) for one eplusout.csv."""
    bldg_id = find_bldg_id_from_path(csv_path)
    if bldg_id is None:
        return (-1, None, None, 0.0, f"skip:no_bldg_id:{csv_path}")

    # Read minimal columns
    try:
        df = pd.read_csv(csv_path, **READ_KW)
    except ValueError as e:
        return (bldg_id, None, None, 0.0, f"missing_cols:{e}")
    except Exception as e:
        return (bldg_id, None, None, 0.0, f"read_fail:{e}")

    ts_col = pick_ts_column(df.columns)
    if ts_col is None:
        return (bldg_id, None, None, 0.0, "no_ts_col")

    # Normalize
    df = df.rename(columns={DATE_COL: "time", ts_col: "ts_elec"})
    df["time"] = parse_time_col(df["time"]).astype("datetime64[ns]")
    df = df.dropna(subset=["time"])
    df["ts_elec"] = pd.to_numeric(df["ts_elec"], errors="coerce").fillna(0.0)

    # 15-min rows for this building
    df15 = df.loc[:, ["time", "ts_elec"]].copy()
    df15.insert(0, "bldg_id", bldg_id)

    # Hourly aggregate for this building
    df_hr = (
        df.assign(time=df["time"].dt.floor("h"))
          .groupby("time", as_index=False)["ts_elec"].sum()
    )
    df_hr.insert(0, "bldg_id", bldg_id)

    ann_kwh = float(df["ts_elec"].sum())
    return (bldg_id, df15, df_hr, ann_kwh, "ok")

# ------------------------
# Discover all CSVs
# ------------------------
csv_paths = []
for bdir in extract_dir.iterdir():
    if bdir.is_dir():
        p = next(bdir.rglob("eplusout.csv"), None)
        if p:
            csv_paths.append(p)
        else:
            print(f"⚠ No eplusout.csv in {bdir}")
print(f"Found {len(csv_paths)} eplusout.csv files")

# Overwrite any existing combined CSVs (remove so headers re-write cleanly)
for p in [OUT_15_CSV, OUT_HR_CSV, OUT_ANL_CSV]:
    if p.exists():
        p.unlink()

# Prepare Parquet writers (if available)
if parquet_ok:
    writer_15 = StreamParquetWriter(
        OUT_15_PQ,
        [("bldg_id", pa.int64()), ("time", pa.timestamp("ns")), ("ts_elec", pa.float64())],
    )
    writer_hr = StreamParquetWriter(
        OUT_HR_PQ,
        [("bldg_id", pa.int64()), ("time", pa.timestamp("ns")), ("ts_elec", pa.float64())],
    )
else:
    writer_15 = writer_hr = None
    print("[warn] pyarrow not available; writing only CSVs. Install: python -m pip install pyarrow")

annual_rows = []

# ------------------------
# Parallel parse + stream writes
# ------------------------
cols_order = ["bldg_id", "time", "ts_elec"]

with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    futs = [pool.submit(process_csv, p) for p in csv_paths]
    for fut in as_completed(futs):
        bldg_id, df15, df_hr, ann_kwh, status = fut.result()
        if status == "ok":
            # Append to CSVs
            append_csv(df15, OUT_15_CSV, cols_order)
            append_csv(df_hr, OUT_HR_CSV, cols_order)
            # Append to Parquet (if available)
            if parquet_ok:
                writer_15.write(df15)
                writer_hr.write(df_hr)
            # Accumulate annual
            annual_rows.append((bldg_id, ann_kwh))
            print(f" {bldg_id} OK")
        elif status == "skipped":
            print(f" {bldg_id} skipped")
        else:
            print(f" {bldg_id} {status}")

# Close parquet writers
if parquet_ok:
    writer_15.close()
    writer_hr.close()

# ------------------------
# Annual combined tables
# ------------------------
anl_df = (
    pd.DataFrame(annual_rows, columns=["bldg_id", "annl_elec"])
      .sort_values("bldg_id")
)
anl_df.to_csv(OUT_ANL_CSV, index=False)

if parquet_ok:
    tbl = pa.Table.from_pandas(anl_df, preserve_index=False)
    pq.write_table(tbl, OUT_ANL_PQ)

print("\nSaved:")
print(f"  15-min  {OUT_15_CSV}")
if parquet_ok: print(f"            {OUT_15_PQ}")
print(f"  Hourly   {OUT_HR_CSV}")
if parquet_ok: print(f"            {OUT_HR_PQ}")
print(f"  Annual   {OUT_ANL_CSV}")
if parquet_ok: print(f"            {OUT_ANL_PQ}")
print("DONE.")


In [ ]:
# AMI DATA PREPARATION FOR OPTIMIZATION

# REQUIRED

# ami_cols = {"id": ami_idx,
#             "id_metadata": ami_meta_idx,
#             "time": "timestamp",
#             "ts_elec": "value",} 
# ami_cols = {"id": "SERVIDEPOINTID",
#            "time": "ENDTIME_EST",
#            "ts_elec": "INTERVAL_READ"}

# CURRENT FORMAT

# SERVICEPOINTID	CHANNELNUMBER	ENDTIME_EST	INTERVAL_READ
# 6000973858	1	6/13/23 14:00	0.057
# 6000973858	1	6/13/23 14:15	0.051


# Path to AMI data

parquet_folder = Path("../data/parquet")
ami = Path("/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/AMI/NREL/consumption")
ami_ind = ami / "ch1_4301002_individual.csv"

df = pd.read_csv(ami_ind)
# parse date time 
df["ENDTIME_EST"] = pd.to_datetime(df["ENDTIME_EST"])
df_2024 = df[df["ENDTIME_EST"].dt.year == 2024]

# total rows
print("Total rows:", len(df_2024))

# total unique service IDs
unique_ids = df_2024["SERVICEPOINTID"].nunique()
print("Total unique service IDs:", unique_ids)

# Define the keys that must be unique
keys = ["SERVICEPOINTID", "ENDTIME_EST"]

# Average duplicates (singles stay the same), write back into INTERVAL_READ
df_2024 = (
    df_2024
    .groupby(keys, as_index=False)["INTERVAL_READ"]
    .mean()
)

# REMOVE FEB 29 FROM AMI DATA SET
df_2024 = df_2024[df_2024["ENDTIME_EST"].dt.date != pd.to_datetime("2024-02-29").date()]


# make hourly data
df_2024_hour = (
    df_2024.set_index("ENDTIME_EST")
           .groupby("SERVICEPOINTID")["INTERVAL_READ"]
           .resample("h").sum()
           .reset_index()
)

out_path_15 = parquet_folder / "ami_ind_data_15.parquet"
df_2024.to_parquet(out_path_15, index=False)
df_2024.to_csv(parquet_folder / "ami_ind_data_15.csv", index=False)

df_2024_hour.to_parquet(parquet_folder / "ami_ind_data_hour.parquet", index=False)
df_2024_hour.to_csv(parquet_folder / "ami_ind_data_hour.csv", index=False)

print("Saved:", out_path_15)
print(df_2024.head())

# 6000974613	1	12/31/24 23:45	0.359
# 6000974626	1	1/1/24 0:00	0.144

In [ ]:
## CHECK FOR DUPLICATES IN AMI

# number of service IDs with more than 1 timeseries (i.e. multiple CHANNELNUMBERs)
service_id_counts = df_2024.groupby("SERVICEPOINTID")["CHANNELNUMBER"].nunique()
multi_ts_ids = (service_id_counts > 1).sum()
print("Number of service IDs with >1 timeseries:", multi_ts_ids)

# --- Check for duplicates ---

keys = ["SERVICEPOINTID", "ENDTIME_EST"]

# Find duplicates
dup_mask = df_2024.duplicated(subset=keys, keep=False)

# Filter duplicates
df_dupes = df_2024[dup_mask].copy()

# Count unique service IDs with duplicates
unique_sid_count = df_dupes["SERVICEPOINTID"].nunique()

print("Number of duplicate rows:", len(df_dupes))
print("Number of service IDs with duplicate rows:", unique_sid_count)

# Save duplicates to CSV
out_csv = parquet_folder / "ami_duplicates_2024.csv"
df_dupes.to_csv(out_csv, index=False)
print("Saved duplicates to:", out_csv)

# --- Average duplicates (collapse to one row per SERVICEPOINTID + ENDTIME_EST) ---
df_2024_avg = (
    df_2024
    .groupby(keys, as_index=False)["INTERVAL_READ"]
    .mean()
    .rename(columns={"INTERVAL_READ": "INTERVAL_READ_avg"})
)

# Sanity check
print("Before rows:", len(df_2024))
print("After rows :", len(df_2024_avg))
print(df_2024_avg.head())


##### CAVEATS AND ASSUMPTIONS
##### The models in Resstock resstock_tmy3_release_2 are not for a leap year. The schedule files do not have a schedule for Feb 29 
##### If run for 2024 in the setRunPeriod measure - throws an error
##### these are run for 2023 in setRunPeriod - BUT are using Ithaca 2024 .epw weather file
##### When running optimization - AMI data for Feb 29 would have to be deleted
##### Only Resstock right now


### Selected counties: 
###792 unique service point IDs, 154 commercial
###ResStock Models: 

###New York State : 33791 models

###Tompkins County : 174 models

###Tompkins and neighboring counties with similar geographic landscape and building construction:

###Tompkins Cortland, Tioga, Chemung, Schuyler, Seneca, Cayuga: 607 models​


In [ ]:
# LOAD PARQUET FILE AND SEE RESULTS

# path to the parquet file AWS
path = Path("/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/NY_parquet_results")
ind_parq = path / "10000-0.parquet"
anl_parq = path / "NY_baseline_metadata_and_annual_results.parquet"

# path to the parquet file pgh
path_pgh = Path(path, "pgh")
ind_parq_pgh = path_pgh / "sample_resstock_ts.parquet"
anl_parq_pgh = path_pgh / "resstock_metro_pgh_results.parquet"

df_1 = pd.read_parquet(ind_parq_pgh)
df_1.to_csv(path_pgh / "ind_parq_pgh.csv", index=False)
print(df_1.shape)         # rows, cols
print(df_1.columns.tolist())

df_2 = pd.read_parquet(anl_parq_pgh)
df_2.to_csv(path_pgh / "anl_parq_pgh.csv", index=False)
print(df_2.shape)         # rows, cols
#print(df_2.columns.tolist())

# df_3 = pd.read_parquet(ind_parq)
# df_3.to_csv(path / "ind_parq.csv", index=False)
# print(df.shape)         # rows, cols
# print(df.columns.tolist())

# df_4 = pd.read_parquet(anl_parq)
# df_4.to_csv(path / "anl_parq.csv", index=False)
# print(df_4.shape)         # rows, cols
# print(df_4.columns.tolist())

print('done')

In [ ]:
# Reference Script

# https://github.com/NREL/buildstock-ami-mapping/blob/e2c-pgh-testing/notebooks/run_model_e2c_pgh.ipynb
# Required fields: 
# cols = {"id": "bldg_id",
#         "time": "timestamp",
#         "ts_elec": "electricity.total.energy_consumption",
#         "annl_elec": "out.electricity.total.energy_consumption"}

# out.electricity.total.energy_consumption.kwh